# A Guided Tour of Ray Core: Multiprocessing Pool

![Anyscale Academy](../images/AnyscaleAcademyLogo.png)

© 2019-2022, Anyscale. All Rights Reserved

[*Distributed multiprocessing.Pool*](https://docs.ray.io/en/latest/multiprocessing.html) makes it easy to scale existing Python applications that use [`multiprocessing.Pool`](https://docs.python.org/3/library/multiprocessing.html) by leveraging *actors*. Ray supports running distributed python programs with the **multiprocessing.Pool** API using Ray Actors, each running on a [workder node](https://docs.ray.io/en/latest/ray-core/actors.html#faq-actors-workers-and-resources), instead of local processes. This makes it easy to scale existing applications that use `multiprocessing.Pool` from a single node to a cluster.

<img src="../images/dist_multi_pool.png" width="70%" height="35%">

First, let's start Ray…

In [1]:
import multiprocessing as mp
import time
import logging
import ray

## Multiprocessing Pool example

The following is a simple Python function with a slight delay added (to make it behave like a more complex calculation)...

In [2]:
# this could be some complicated and compute intensive task
def func(x):
    time.sleep(1.5)
    return x ** 2

Then, use the Ray's drop-in replacement for [multiprocessing pool](https://docs.ray.io/en/latest/multiprocessing.html)

In [3]:
ray.init(
    ignore_reinit_error=True,
    logging_level=logging.ERROR,
)

/Users/ravinkohli/self-attention-from-scratch/academy/.venv/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.12.10
Ray version:,2.55.1
Dashboard:,http://127.0.0.1:8265


Now we'll create a *Pool* using and distribute its tasks across a cluster (or across the available cores on a laptop):

In [4]:
%%time

from ray.util.multiprocessing import Pool

pool = Pool()

for result in pool.map(func, range(10)):
    print(result)

0
1
4
9
16
25
36
49
64
81
CPU times: user 204 ms, sys: 100 ms, total: 304 ms
Wall time: 3.01 s


The distributed version has the trade-off of increased overhead, although now it can scale-out horizontally across a cluster. The benefits would be more pronounced with a more computationally expensive calculation.

In [5]:
pool.terminate()

Let's define a compute intensive class that does some matrix computation. Consider this could be a compute intenstive task doing massive tensor transformation or computation.

In [7]:
import numpy as np
def task(n):
    # Simulate a long intensive task
    #TODO
    array = np.random.rand(1000, 1000)
    result = np.dot(array, array)
    # do some matrix computation 
    # and return results
    return result

Define a Ray remote task that launches task() across a pool of Actors on the cluster. It creates a pool of Ray Actors, each scheduled on a cluster worker.


In [12]:
@ray.remote
def launch_long_running_tasks(num_pool):
    # Doing the work, collecting data, updating the database
    # create an Actor pool of num_pool workers nodes
    pool = Pool(num_pool)
    results = []
    # Iterate over 50 times in batches of 10
    for result in pool.map(task, range(1, 50, 10)):
        results.append(result)
        
    # Done so terminate pool
    pool.terminate()
    
    return results

### Create a Actor like supervisor that launches all these remote tasks


In [13]:
@ray.remote
class LaunchDistributedTasks:
    def __init__(self, limit=5):
        self._limit = limit

    def launch(self):
        # launch the remote task
        return launch_long_running_tasks.remote(self._limit)

### Launch our supervisor

In [14]:
hdl = LaunchDistributedTasks.remote(5)
print("Launched remote jobs")

Launched remote jobs


### Launched remote jobs

In [15]:
values = ray.get(ray.get(hdl.launch.remote()))
print(f" list of results :{values}")
print(f" Total results: {len(values)}")

 list of results :[array([[252.64877566, 249.74211134, 242.53332531, ..., 248.52412546,
        244.44619227, 250.3983107 ],
       [249.92044519, 245.57413528, 242.1595454 , ..., 246.0564352 ,
        241.43563365, 243.68391382],
       [255.01314823, 254.29406431, 247.28643855, ..., 248.05736058,
        245.47772026, 250.57866071],
       ...,
       [258.65248019, 247.84424975, 245.9228383 , ..., 253.32422123,
        244.26835   , 249.95685532],
       [256.17582379, 248.76913206, 244.74882604, ..., 246.9224063 ,
        244.12826216, 246.27601287],
       [251.07537583, 246.94976333, 243.58676203, ..., 242.34140996,
        235.62005837, 246.2008136 ]], shape=(1000, 1000)), array([[243.55241982, 249.5203004 , 247.06444666, ..., 244.4269091 ,
        252.84931468, 245.89181412],
       [250.92367376, 252.73273284, 248.57034229, ..., 243.28120476,
        246.60867606, 251.54002545],
       [252.4670806 , 251.40330542, 250.64746862, ..., 249.34300991,
        249.12622692, 250.6250

Finally, shutdown Ray

In [16]:
ray.shutdown()

### Excercises
1. Can you convert task() into a complicated function?
2. Use `task()` in pool.map(task,....)

### Homework
1. Write a Python multiprocessing.pool version of task() and compare the timings with the Ray distributed multiprocessing.pool.
2. Do you see a difference in timings?
3. Write a distributed crawler that downloads gifs or pdfs